In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "5"
import torch.nn.functional as F
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch
import os,sys
current_file = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(current_file), '..')))

In [2]:
from dataset.qm9s_dataset import Multi_process_Qm9sDataset
from dataset.nist_dataset import Multi_process_NISTDataset
from dataset.collate_fn import Multi_process_batch_collate_fn
from torch.utils.data import DataLoader

/home/zjh/miniconda3/envs/mol_spectra/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
import faiss

def compute_recall_with_faiss(all_mol_feats, all_spectra_feats, tops):
    N, d = all_mol_feats.shape
    assert all_spectra_feats.shape == (N, d)
    
    # --- L2 归一化 ---
    all_mol_feats = all_mol_feats / torch.norm(all_mol_feats, dim=1, keepdim=True)
    all_spectra_feats = all_spectra_feats / torch.norm(all_spectra_feats, dim=1, keepdim=True)
    
    mol_np = all_mol_feats.cpu().numpy()
    spectra_np = all_spectra_feats.cpu().numpy()
    
    target_indices = torch.arange(N).unsqueeze(1)  # [N, 1]
    k_max = max(tops)
    
    recall_m2s, recall_s2m = [], []
    
    # --- mol -> spectra ---
    index_m2s = faiss.IndexFlatIP(d)
    index_m2s.add(spectra_np)
    D_m2s, I_m2s = index_m2s.search(mol_np, k_max)  
    
    for k in tops:
        topk_indices = torch.tensor(I_m2s[:, :k])
        correct = (topk_indices == target_indices).any(dim=1).float()
        recall_m2s.append(correct.mean().item())
    
    # --- spectra -> mol ---
    index_s2m = faiss.IndexFlatIP(d)
    index_s2m.add(mol_np)
    D_s2m, I_s2m = index_s2m.search(spectra_np, k_max)
    
    for k in tops:
        topk_indices = torch.tensor(I_s2m[:, :k])
        correct = (topk_indices == target_indices).any(dim=1).float()
        recall_s2m.append(correct.mean().item())
    
    # --- 返回结果 ---
    results = {
        "recall_m2s": recall_m2s,
        "recall_s2m": recall_s2m
    }
    return results


In [6]:
import yaml
qm9s = "/home/zjh/final-ssmoe-mvms/config/qm9s_retrieval.yaml"
nist = "/config/nist_retrieval.yaml"
with open(qm9s, "r") as f:
    config = yaml.safe_load(f)
dataset = Multi_process_Qm9sDataset(db_path=config["test_db_path"],dict_path=config["dict_path"],config=config)
dataloader = DataLoader(dataset, batch_size=256, shuffle=False,collate_fn=Multi_process_batch_collate_fn)

In [10]:
from model.ssmoe_mvms_retrieval import SSMoE_MVMS_Retrieval
from model.model_config import MolConfig

mol_config = MolConfig(mol_pretrain_pth=config["mol_pretrain_pth"],mol_dict_pth=config["mol_dict_pth"])
model = SSMoE_MVMS_Retrieval(config,mol_config)

"""
Load checkpoint
"""
qm9s_ir_retrieval_ckpt = "/home/zjh/final-ssmoe-mvms/ckpt/retrieval/QM9s/ir_retrieval.pt"
qm9s_raman_retrieval_ckpt = "/home/zjh/final-ssmoe-mvms/ckpt/retrieval/QM9s/raman_retrieval.pt"
nist_ir_retrieval_ckpt = "/ckpt/retrieval/NIST/ir_retrieval.pt"


all_spectra_feats = []
all_mol_feats = []
mask_ratio = config["mask_ratio"]
ckpt = torch.load(qm9s_raman_retrieval_ckpt,map_location='cpu',weights_only=False)
model.load_state_dict(ckpt['model'])
model = model.eval()
model = model.cuda()
with torch.no_grad():
    for batch_list in tqdm(dataloader,total=len(dataloader)):
        src_tokens,src_coord,src_distance,src_edge_type,spectra,smi = batch_list['unimol']["src_tokens"].cuda(), batch_list['unimol']["src_coord"].cuda(), batch_list['unimol']["src_distance"].cuda(), batch_list['unimol']["src_edge_type"].cuda(), batch_list['unimol']["raman"].cuda(),batch_list['unimol']["smi"]
        x,edge_index,edge_attr,batch = batch_list['pyg']["x"].cuda(), batch_list['pyg']["edge_index"].cuda(), batch_list['pyg']["edge_attr"].cuda(),batch_list['pyg'].batch.cuda()
        _,mol_feats,spectra_feats = model(src_tokens, src_coord, src_distance, src_edge_type, smi, x, edge_index, edge_attr, batch, spectra, "cuda", mask_ratio,preserve_flag=[0,1,2,3])
        all_spectra_feats.append(spectra_feats)
        all_mol_feats.append(mol_feats) 
all_spectra_feats = torch.cat(all_spectra_feats,dim=0)
all_mol_feats = torch.cat(all_mol_feats,dim=0)

tops = [1,2,3,4,5]
results = compute_recall_with_faiss(all_mol_feats, all_spectra_feats, tops)

model name: /home/zjh/final-ssmoe-mvms/model/SMILES/MoLFormer
number of SMiles_Encoder parameters: 44.38M
model name: /home/zjh/final-ssmoe-mvms/model/SMILES/Chemberta
number of SMiles_Encoder parameters: 44.10M
Loading from /home/zjh/final-ssmoe-mvms/model/graph_mvp/model_g.pth ...
GraphMVP missing_keys:[]
GraphMVP unexpeted_keys:[]
number of Unimol parameters: 47.33M
Loading pretrained weights from /home/zjh/final-ssmoe-mvms/model/unimol/unimol.pt
missing_keys:  []
unexpeted_keys:  ['lm_head.weight', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'pair2coord_proj.linear1.weight', 'pair2coord_proj.linear1.bias', 'pair2coord_proj.linear2.weight', 'pair2coord_proj.linear2.bias', 'dist_head.dense.weight', 'dist_head.dense.bias', 'dist_head.layer_norm.weight', 'dist_head.layer_norm.bias', 'dist_head.out_proj.weight', 'dist_head.out_proj.bias', 'encoder.final_head_layer_norm.weight', 'encoder.final_head_layer_norm.bias'

100%|██████████| 51/51 [00:25<00:00,  2.03it/s]
